# Hindi Speech HMM and Dialect Pattern Analysis — LAHAJA Dataset

## Cell 1 — Install libraries

In [ ]:
%pip -q install hmmlearn librosa soundfile pyarrow scikit-learn mlflow

## Cell 2 — Import libraries and set parameters

In [ ]:
import os
import io
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import librosa
import pyarrow.parquet as pq
from tqdm.auto import tqdm
from hmmlearn.hmm import GaussianHMM
from scipy.special import logsumexp
from google.colab import drive
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
np.random.seed(42)

N_MFCC = 13
MAX_FRAMES = 60
N_STATES = 4
HMM_ITERATIONS = 10
SAMPLE_RATE = 16000

## Cell 3 — Connect Google Drive and find dataset files

In [ ]:
drive.mount("/content/drive")

required_files = [
    "test-00000-of-00004.parquet",
    "test-00001-of-00004.parquet",
    "test-00002-of-00004.parquet",
    "test-00003-of-00004.parquet"
]

found_files = {}

for root, dirs, files in os.walk("/content/drive"):
    for filename in files:
        if filename in required_files:
            found_files[filename] = os.path.join(root, filename)

for filename in required_files:
    print(filename, "->", found_files.get(filename, "NOT FOUND"))

if len(found_files) != 4:
    raise FileNotFoundError("Could not find all four LAHAJA Parquet shards in Google Drive.")

## Cell 4 — Load dataset metadata

In [ ]:
metadata_columns = [
    "duration", "scenario", "fname", "native_language", "gender",
    "age_group", "native_state", "native_district", "sp_id", "lang"
]

metadata_parts = []

for filename in required_files:
    part = pd.read_parquet(found_files[filename], columns=metadata_columns)
    part["source_shard"] = filename
    metadata_parts.append(part)

metadata = pd.concat(metadata_parts, ignore_index=True)

print("Metadata shape:", metadata.shape)
print("Unique language tags:", metadata["lang"].dropna().unique())
print("Unique speakers:", metadata["sp_id"].nunique())
display(metadata.head())
display(metadata[["duration", "native_state", "scenario"]].describe(include="all").T)

## Cell 5 — Explore regional sample distribution

In [ ]:
state_counts = metadata["native_state"].value_counts().sort_values()

plt.figure(figsize=(10, 7))
state_counts.plot(kind="barh")
plt.xlabel("Number of recordings")
plt.ylabel("Native state")
plt.title("LAHAJA recordings by native state")
plt.tight_layout()
plt.show()

print("Total recordings in metadata:", len(metadata))
print("Total duration in hours:", round(metadata["duration"].sum() / 3600, 2))

## Cell 6 — Define MFCC feature extractor

In [ ]:
def extract_mfcc_sequence(audio_object):
    if audio_object is None:
        raise ValueError("Audio object is missing")

    if hasattr(audio_object, "as_py"):
        audio_object = audio_object.as_py()

    audio_bytes = audio_object.get("bytes")
    if audio_bytes is None:
        raise ValueError("Audio bytes are missing")

    y, sr = librosa.load(io.BytesIO(audio_bytes), sr=SAMPLE_RATE, mono=True)

    if len(y) == 0:
        raise ValueError("Audio is empty")

    mfcc = librosa.feature.mfcc(
        y=y,
        sr=sr,
        n_mfcc=N_MFCC,
        n_fft=1024,
        hop_length=512
    ).T

    if len(mfcc) > MAX_FRAMES:
        selected = np.linspace(0, len(mfcc) - 1, MAX_FRAMES).astype(int)
        mfcc = mfcc[selected]

    return mfcc.astype(np.float32)

## Cell 7 — Extract MFCC sequences

In [ ]:
sequences = []
sequence_rows = []
failed_rows = []
processed = 0
start_time = time.time()

for filename in required_files:
    parquet_file = pq.ParquetFile(found_files[filename])
    metadata_part = pd.read_parquet(found_files[filename], columns=metadata_columns)

    row_index = 0
    for batch in parquet_file.iter_batches(batch_size=1, columns=["audio_filepath"]):
        audio_object = batch.column(0)[0].as_py()
        row = metadata_part.iloc[row_index].to_dict()

        try:
            sequence = extract_mfcc_sequence(audio_object)
            sequences.append(sequence)
            sequence_rows.append(row)
        except Exception as error:
            failed_rows.append({
                "source_shard": filename,
                "row_index": row_index,
                "error": str(error)
            })

        row_index += 1
        processed += 1

        if processed % 250 == 0:
            print(f"Processed {processed} recordings")

sequence_metadata = pd.DataFrame(sequence_rows).reset_index(drop=True)
failed_df = pd.DataFrame(failed_rows)

print("Attempted recordings:", processed)
print("Successful sequences:", len(sequences))
print("Failed recordings:", len(failed_df))
print("Elapsed minutes:", round((time.time() - start_time) / 60, 2))
print("First sequence shape:", sequences[0].shape if sequences else "No sequences")
display(failed_df.head())

## Cell 8 — Visualize one MFCC sequence

In [ ]:
sample_sequence = sequences[0]

plt.figure(figsize=(12, 4))
plt.imshow(sample_sequence.T, aspect="auto", origin="lower")
plt.xlabel("Frame index")
plt.ylabel("MFCC coefficient")
plt.title("MFCC sequence from one LAHAJA recording")
plt.colorbar(label="MFCC value")
plt.tight_layout()
plt.show()

print("Frames:", sample_sequence.shape[0])
print("MFCC dimensions:", sample_sequence.shape[1])

## Cell 9 — Prepare sequences for the HMM

In [ ]:
valid_indices = [i for i, seq in enumerate(sequences) if len(seq) >= 2]
valid_sequences = [sequences[i] for i in valid_indices]
lengths = [len(seq) for seq in valid_sequences]
X_hmm = np.concatenate(valid_sequences, axis=0)

print("Number of recordings used:", len(valid_sequences))
print("Combined frame matrix:", X_hmm.shape)
print("Total sequence lengths:", sum(lengths))
print("Sequence count matches lengths:", len(lengths) == len(valid_sequences))

## Cell 10 — Train the Gaussian HMM

In [ ]:
hmm_model = GaussianHMM(
    n_components=N_STATES,
    covariance_type="diag",
    n_iter=HMM_ITERATIONS,
    random_state=42,
    verbose=True,
    min_covar=1e-3
)

hmm_model.fit(X_hmm, lengths)

print("HMM training completed")
print("Converged:", hmm_model.monitor_.converged)
print("Iterations:", hmm_model.monitor_.iter)
print("Start probabilities:", hmm_model.startprob_)
print("Transition matrix:")
display(pd.DataFrame(hmm_model.transmat_))

## Cell 11 — Calculate HMM likelihood scores

In [ ]:
log_likelihoods = []
mean_log_likelihoods = []
frame_counts = []

for sequence in tqdm(valid_sequences):
    score = hmm_model.score(sequence)
    log_likelihoods.append(score)
    mean_log_likelihoods.append(score / len(sequence))
    frame_counts.append(len(sequence))

results = sequence_metadata.iloc[valid_indices].reset_index(drop=True).copy()
results["frame_count"] = frame_counts
results["hmm_log_likelihood"] = log_likelihoods
results["mean_log_likelihood"] = mean_log_likelihoods

print("Scored recordings:", len(results))
display(results[["native_state", "duration", "frame_count", "mean_log_likelihood"]].head())
display(results["mean_log_likelihood"].describe().to_frame("Mean log likelihood per frame"))

## Cell 12 — Forward algorithm

In [ ]:
def forward_log_likelihood(model, sequence):
    log_emissions = model._compute_log_likelihood(sequence)
    log_start = np.log(model.startprob_ + 1e-300)
    log_transition = np.log(model.transmat_ + 1e-300)

    alpha = np.zeros_like(log_emissions)
    alpha[0] = log_start + log_emissions[0]

    for t in range(1, len(sequence)):
        alpha[t] = log_emissions[t] + logsumexp(
            alpha[t - 1][:, None] + log_transition,
            axis=0
        )

    return logsumexp(alpha[-1])

sample_score = hmm_model.score(sample_sequence)
forward_score = forward_log_likelihood(hmm_model, sample_sequence)

print("HMM score:", sample_score)
print("Forward algorithm score:", forward_score)
print("Absolute difference:", abs(sample_score - forward_score))

## Cell 13 — Backward algorithm

In [ ]:
def backward_log_likelihood(model, sequence):
    log_emissions = model._compute_log_likelihood(sequence)
    log_transition = np.log(model.transmat_ + 1e-300)
    log_start = np.log(model.startprob_ + 1e-300)

    beta = np.zeros_like(log_emissions)

    for t in range(len(sequence) - 2, -1, -1):
        beta[t] = logsumexp(
            log_transition + log_emissions[t + 1][None, :] + beta[t + 1][None, :],
            axis=1
        )

    return logsumexp(log_start + log_emissions[0] + beta[0])

backward_score = backward_log_likelihood(hmm_model, sample_sequence)

print("HMM score:", sample_score)
print("Backward algorithm score:", backward_score)
print("Absolute difference:", abs(sample_score - backward_score))

## Cell 14 — Viterbi algorithm

In [ ]:
viterbi_states = hmm_model.predict(sample_sequence)

plt.figure(figsize=(12, 3))
plt.step(np.arange(len(viterbi_states)), viterbi_states, where="mid")
plt.xlabel("Frame index")
plt.ylabel("Hidden state")
plt.title("Viterbi hidden-state path for one recording")
plt.yticks(range(N_STATES))
plt.tight_layout()
plt.show()

print("Predicted hidden states:", np.unique(viterbi_states))
print("State path length:", len(viterbi_states))

## Cell 15 — Compare HMM scores across regions

In [ ]:
state_summary = (
    results.groupby("native_state")
    .agg(
        recordings=("mean_log_likelihood", "size"),
        mean_score=("mean_log_likelihood", "mean"),
        median_score=("mean_log_likelihood", "median"),
        std_score=("mean_log_likelihood", "std")
    )
    .query("recordings >= 50")
    .sort_values("mean_score")
)

display(state_summary)

plt.figure(figsize=(10, 7))
state_summary["mean_score"].plot(kind="barh", xerr=state_summary["std_score"])
plt.xlabel("Mean log likelihood per frame")
plt.ylabel("Native state")
plt.title("HMM acoustic sequence fit by native state")
plt.tight_layout()
plt.show()

## Cell 16 — Visualize regional score distributions

In [ ]:
states_to_plot = state_summary.index.tolist()
plot_data = results[results["native_state"].isin(states_to_plot)].copy()

plt.figure(figsize=(13, 6))
plot_data.boxplot(column="mean_log_likelihood", by="native_state", rot=75, grid=False)
plt.xlabel("Native state")
plt.ylabel("Mean log likelihood per frame")
plt.title("Distribution of HMM likelihood by native state")
plt.suptitle("")
plt.tight_layout()
plt.show()

## Cell 17 — Compare score and recording conditions

In [ ]:
results["duration"] = pd.to_numeric(results["duration"], errors="coerce")

print("Correlation between duration and mean HMM likelihood:")
print(results[["duration", "mean_log_likelihood"]].corr().iloc[0, 1])

scenario_summary = (
    results.groupby("scenario")["mean_log_likelihood"]
    .agg(["count", "mean", "median", "std"])
    .sort_values("mean")
)

display(scenario_summary)

plt.figure(figsize=(7, 4))
results.boxplot(column="mean_log_likelihood", by="scenario", grid=False)
plt.xlabel("Speech scenario")
plt.ylabel("Mean log likelihood per frame")
plt.title("HMM likelihood by speech scenario")
plt.suptitle("")
plt.tight_layout()
plt.show()

## Cell 18 — Discover dialect-like acoustic groups

In [ ]:
state_paths = [hmm_model.predict(sequence) for sequence in valid_sequences]
state_features = []

for path in state_paths:
    occupancy = np.bincount(path, minlength=N_STATES) / len(path)
    transitions = np.zeros((N_STATES, N_STATES), dtype=np.float32)
    for first, second in zip(path[:-1], path[1:]):
        transitions[first, second] += 1
    row_sums = transitions.sum(axis=1, keepdims=True)
    transitions = np.divide(transitions, row_sums, out=np.zeros_like(transitions), where=row_sums != 0)
    state_features.append(np.concatenate([occupancy, transitions.ravel()]))

state_features = np.asarray(state_features, dtype=np.float32)
state_feature_names = [f"state_{i}_occupancy" for i in range(N_STATES)]
state_feature_names += [f"transition_{i}_{j}" for i in range(N_STATES) for j in range(N_STATES)]
state_feature_df = pd.DataFrame(state_features, columns=state_feature_names)
state_feature_df["native_state"] = results["native_state"].fillna("Unknown").to_numpy()
state_feature_df["native_district"] = results["native_district"].fillna("Unknown").to_numpy()
state_feature_df["native_language"] = results["native_language"].fillna("Unknown").to_numpy()

regional_state_features = state_feature_df.groupby("native_state").filter(lambda group: len(group) >= 50)
regional_summary = regional_state_features.groupby("native_state")[state_feature_names].mean()
display(regional_summary)

occupancy_columns = [f"state_{i}_occupancy" for i in range(N_STATES)]
regional_summary[occupancy_columns].plot(kind="bar", figsize=(12, 5))
plt.xlabel("Speaker native state")
plt.ylabel("Mean proportion of frames assigned to HMM state")
plt.title("HMM hidden-state occupancy by native state")
plt.tight_layout()
plt.show()

state_feature_df["mean_log_likelihood"] = results["mean_log_likelihood"].to_numpy()
display(state_feature_df.groupby("native_state").agg(recordings=("mean_log_likelihood", "size"), mean_score=("mean_log_likelihood", "mean")).query("recordings >= 50").sort_values("mean_score"))
print("These are candidate regional acoustic patterns, not verified Hindi dialect labels.")

## Cell 19 — Compare HMM configurations with MLflow

In [ ]:
import mlflow
from sklearn.model_selection import train_test_split

mlflow.set_tracking_uri("file:./mlruns")
mlflow.set_experiment("LAHAJA_Hindi_HMM_Experiments")

speaker_ids = sequence_metadata.iloc[valid_indices]["sp_id"].fillna("Unknown").astype(str).to_numpy()
unique_speakers = np.unique(speaker_ids)
train_speakers, test_speakers = train_test_split(unique_speakers, test_size=0.2, random_state=42)
train_indices = [i for i, speaker in enumerate(speaker_ids) if speaker in set(train_speakers)]
test_indices = [i for i, speaker in enumerate(speaker_ids) if speaker in set(test_speakers)]
train_indices = train_indices[:1200]
test_indices = test_indices[:300]

train_sequences = [valid_sequences[i] for i in train_indices]
test_sequences = [valid_sequences[i] for i in test_indices]
train_lengths = [len(sequence) for sequence in train_sequences]
train_matrix = np.concatenate(train_sequences, axis=0)

experiment_results = []

for n_states in [3, 4, 5]:
    with mlflow.start_run(run_name=f"GaussianHMM_{n_states}_states"):
        candidate_model = GaussianHMM(
            n_components=n_states,
            covariance_type="diag",
            n_iter=8,
            random_state=42,
            min_covar=1e-3
        )
        candidate_model.fit(train_matrix, train_lengths)
        test_scores = [candidate_model.score(sequence) / len(sequence) for sequence in test_sequences]
        mean_test_score = float(np.mean(test_scores))
        mlflow.log_param("n_states", n_states)
        mlflow.log_param("covariance_type", "diag")
        mlflow.log_param("n_iter", 8)
        mlflow.log_param("train_recordings", len(train_sequences))
        mlflow.log_param("test_recordings", len(test_sequences))
        mlflow.log_metric("mean_test_log_likelihood_per_frame", mean_test_score)
        mlflow.log_metric("converged", int(candidate_model.monitor_.converged))
        mlflow.log_metric("iterations", int(candidate_model.monitor_.iter))
        experiment_results.append({
            "n_states": n_states,
            "mean_test_log_likelihood_per_frame": mean_test_score,
            "converged": candidate_model.monitor_.converged,
            "iterations": candidate_model.monitor_.iter
        })

experiment_table = pd.DataFrame(experiment_results).sort_values("mean_test_log_likelihood_per_frame", ascending=False)
display(experiment_table)
print("MLflow tracking files are stored in the notebook runtime under ./mlruns, not Google Drive.")

## Cell 20 — Display final scores

In [ ]:
print("HMM score on all processed sequences:", hmm_model.score(X_hmm, lengths))
print("Mean log-likelihood per frame:", np.mean(mean_log_likelihoods))
print("Successfully processed recordings:", len(valid_sequences))
